In [2]:
import pandas as pd
import gc
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error

In [3]:
df_weekly_final=pd.read_csv("C:/Users/shrut/Desktop/walmart/walmart_weekly_perfect.csv")

In [4]:
grouped = df_weekly_final.groupby(['item_id', 'store_id'])

# 1. Lags banana (Using explicit group shift)
df_weekly_final['lag_1'] = grouped['sales'].shift(1).astype('float32')
df_weekly_final['lag_2'] = grouped['sales'].shift(2).astype('float32')
df_weekly_final['lag_4'] = grouped['sales'].shift(4).astype('float32')

# 2. Rolling Means banana (Using clean window transforms inside groups)
df_weekly_final['rolling_mean_4'] = grouped['sales'].transform(lambda x: x.shift(1).rolling(4).mean()).astype('float32')
df_weekly_final['rolling_mean_12'] = grouped['sales'].transform(lambda x: x.shift(1).rolling(12).mean()).astype('float32')

# Rows remove karna jinke paas unka apna valid pichla itihaas nahi hai
df_weekly_final.dropna(subset=['lag_1', 'lag_2', 'lag_4', 'rolling_mean_4', 'rolling_mean_12'], inplace=True)

gc.collect()
df_weekly_final.head(5)

,item_id,store_id,date,sales,is_holiday,snap_active,sell_price,weekend_sales_pct,lag_1,lag_2,lag_4,rolling_mean_4,rolling_mean_12
12,0,0,2011-04-18,3,1,0,2.0,33.333333,9.0,8.0,8.0,6.50,8.333333
13,0,0,2011-04-25,5,1,1,2.0,100.000000,3.0,9.0,1.0,5.25,8.333333
14,0,0,2011-05-02,17,1,1,2.0,0.000000,5.0,3.0,8.0,6.25,8.000000
15,0,0,2011-05-09,8,0,1,2.0,50.000000,17.0,5.0,9.0,8.50,8.833333
16,0,0,2011-05-16,9,0,0,2.0,44.444444,8.0,17.0,3.0,8.25,8.666667


In [5]:

features = ['item_id', 'store_id','is_holiday', 'snap_active',
            'lag_1','lag_4','lag_2', 'rolling_mean_4','weekend_sales_pct','rolling_mean_12','sell_price']

# Jo target model ko predict karna hai
target = 'sales'
df_weekly_final['date']=pd.to_datetime(df_weekly_final['date'])
max_date=df_weekly_final['date'].max()
split_date = max_date - pd.Timedelta(weeks=4)

print(f"-> Split Date: {split_date.date()}")

train_mask = df_weekly_final['date'] <= split_date
test_mask = df_weekly_final['date'] > split_date

X_train, y_train = df_weekly_final[train_mask][features], df_weekly_final[train_mask][target]
X_test, y_test = df_weekly_final[test_mask][features], df_weekly_final[test_mask][target]


# Fast CPU training ke liye settings optimize ki hain
model_xgb = xgb.XGBRegressor(
    n_estimators=100,       # Kitne trees banane hain
    learning_rate=0.05,     # Seekhne ki raftar (eta)
    max_depth=10,            # Tree ki gehrayi
    random_state=42,
    n_jobs=-1               # Laptop ke saare CPU cores use karne ke liye
)

# Training shuru
model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=10              # Har 10 tree ke baad progress dikhayega
) 

# Test data par predictions save karna
df_weekly_final.loc[test_mask, 'predicted_sales'] = model_xgb.predict(X_test)

# Negative values ko 0 karna (kyunki sales kabhi minus mein nahi ho sakti)
df_weekly_final.loc[df_weekly_final['predicted_sales'] < 0, 'predicted_sales'] = 0

rmse = np.sqrt(mean_squared_error(y_test, df_weekly_final[test_mask]['predicted_sales']))
gc.collect()

-> Split Date: 2016-03-21
[0]	validation_0-rmse:20.80412
[10]	validation_0-rmse:13.36041
[20]	validation_0-rmse:9.56723
[30]	validation_0-rmse:7.87412
[40]	validation_0-rmse:7.22471
[50]	validation_0-rmse:6.99158
[60]	validation_0-rmse:6.92105
[70]	validation_0-rmse:6.90011
[80]	validation_0-rmse:6.89111
[90]	validation_0-rmse:6.87984
[99]	validation_0-rmse:6.87999


53

In [6]:
y_true=df_weekly_final[test_mask]['sales'].values
y_pred=df_weekly_final[test_mask]['predicted_sales'].values
def calculate_forecast_metrics(y_true , y_pred):
    y_true=np.array(y_true,dtype=np.float64)
    y_pred=np.array(y_pred,dtype=np.float64)
    bias=np.mean(y_true-y_pred)#positive ya negative

    mask=y_true!=0
    if np.sum(mask)>0:
        mape=np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100
    else:
        mape=np.mean
    total_actual_sales=np.sum(y_true)
    if total_actual_sales!=0:
        wmape=(np.sum(np.abs(y_true-y_pred))/total_actual_sales)*100
    else:
        wmape=np.nan
    forecast_error=wmape if not np.isnan(wmape) else np.nan
    forecast_accuracy=100-forecast_error if not np.isnan(forecast_error) else np.nan
    metrics={
        "bias":round(bias,4),
        "mape":round(mape,2) if not np.isnan(mape) else "na(all actuals are zero)",
        "wmape":round(wmape,2),
        "overall forecast error":round(forecast_error,2),
        "overall forecast accuracy":round(forecast_accuracy,2)}
    return metrics
calculate_forecast_metrics(y_true,y_pred)

{'bias': np.float64(-0.0972),
 'mape': np.float64(38.34),
 'wmape': np.float64(27.42),
 'overall forecast error': np.float64(27.42),
 'overall forecast accuracy': np.float64(72.58)}

In [7]:
output="C:/Users/shrut/Desktop/walmart/predictions.csv"
df_weekly_final.to_csv(output, index=False)
df_weekly_final.head(5)

,item_id,store_id,date,sales,is_holiday,snap_active,sell_price,weekend_sales_pct,lag_1,lag_2,lag_4,rolling_mean_4,rolling_mean_12,predicted_sales
12,0,0,2011-04-18,3,1,0,2.0,33.333333,9.0,8.0,8.0,6.50,8.333333,NaN
13,0,0,2011-04-25,5,1,1,2.0,100.000000,3.0,9.0,1.0,5.25,8.333333,NaN
14,0,0,2011-05-02,17,1,1,2.0,0.000000,5.0,3.0,8.0,6.25,8.000000,NaN
15,0,0,2011-05-09,8,0,1,2.0,50.000000,17.0,5.0,9.0,8.50,8.833333,NaN
16,0,0,2011-05-16,9,0,0,2.0,44.444444,8.0,17.0,3.0,8.25,8.666667,NaN
